# Case set up
### Import libraries

In [ ]:
import os
import sys
import copy
import time
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib as jl
threads = 4

from tqdm import tqdm
from sklearn.model_selection import train_test_split

Visual set up

In [ ]:
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
%matplotlib qt5

# Use custom style
style_name = 'paper'
mpl.style.use(
    os.environ['WORKSPACE_PATH'] + '/RESEARCH/styles/matplotlib/' + style_name + '.mplstyle'
) # See: https://matplotlib.org/1.5.3/users/style_sheets.html

In [ ]:
# Get colors list
colors = mpl.rcParams['axes.prop_cycle'].by_key()['color']*10

# Figure/animation saving parameters
# mpl.rcParams['savefig.dpi'] = 100

# Font parameters
mpl.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
# mpl.rcParams['text.latex.preamble'] = join([
#     r'\usepackage{amsmath}',
#     r'\boldmath',
#     r'\usepackage{pifont}',
#     r'\usepackage{tikz}',
#     r'\usepackage[utf8]{inputenc}'
# ])
# mpl.rcParams['font.family'] = 'lmodern'

# # Tick parameters
# mpl.rcParams['xtick.labelsize'] = 40
# mpl.rcParams['ytick.labelsize'] = 40
# mpl.rcParams['xtick.major.size'] = 20
# mpl.rcParams['xtick.major.width'] = 2
# mpl.rcParams['xtick.minor.size'] = 10
# mpl.rcParams['xtick.minor.width'] = 2

# # Tick parameters
# mpl.rcParams['axes.labelsize'] = 40
# mpl.rcParams['axes.titlesize'] = 40
# mpl.rcParams['legend.fontsize'] = 35


### Define PrODE and case (model) directories paths

In [ ]:
prode_dir = os.environ['WORKSPACE_PATH'] + '/ML/PrODE/prode/'
case_dir = "../trained/"

In [ ]:
if importlib.util.find_spec('prode') is None:
    sys.path.append(os.path.normpath(prode_dir))
sys.path.append(os.path.normpath(case_dir))
cwd = os.getcwd()

### Import additional libraries

In [ ]:
# PrODE -----------------------------------------------------------------------
from prode.utilities import utils
from prode.apps import functionObjects as fun
from prode.postprocessing import testing, training

# Extra modules ---------------------------------------------------------------
# Inputs
import inputs
# Physics-related
from physics.data_generator import DataGenerator
from prode.physics.general import const
from prode.physics.thermochemistry import specie as specieMdl
from prode.physics.thermochemistry.reactionThermo import mixture as mixMdl

Change directory

In [ ]:
os.chdir(os.path.normpath(case_dir))

### Initialize network

In [ ]:
net = fun._net(
    input_module=inputs
)

### Initialize PrODE model

In [ ]:
from inputs.general import flags

for k in ('train', 'save'):
    flags[k] = 0
if flags['load'] not in (1, 2):
    utils.raise_value_err(
        "The network has to be restored from a saved or a checkpointed "
        "one. Modified the 'load' entry in the `flags` input variable."
    )
model = fun._model(
    flags=flags,
    input_module=inputs,
    net=net
)

### Generate testing data
Choose specific test scenarios

In [ ]:
indices = np.array([25, 27, 100]).T

In [ ]:
# Generate Testing Data ---------------------------------------------------
dataGen_args = utils.get_args_values(
    args=utils.get_args(DataGenerator.__init__),
    modules=[inputs.data, inputs.general]
)
dataGen = DataGenerator(
    **dataGen_args
)
dataGen.data_dir = case_dir + "/data/"
if not os.path.exists(dataGen.data_dir):
    os.makedirs(dataGen.data_dir)

utils.print_main("Generating testing pts ...")
if inputs.data.from_file:
    test_data = dataGen.testing_data(
        inputs.data.file['test'],
        header=inputs.data.header,
        indices=indices-1
    )
else:
    raise NotImplementedError

Change directory

In [ ]:
os.chdir(os.path.normpath(cwd))

# Postprocessing - Model analysis

## Set up analysis

In [ ]:
path = cwd + '/outputs/'
if not os.path.exists(path):
    os.makedirs(path)
    
FLAGS = {
    'pred': 0,
    'input_space': 1,
    'one_pass': 0,
    'accuracy': 0,
    'comp_time': 0,
    'speedup': 0,
    'recursive': 0,
    'noneq_dist': 0,
    'levels': 0,
    'animations': 0,
    'adaptive': 0,
    'park_ts1': 0
}

#### Get predictions

In [ ]:
if FLAGS['pred']:
    utils.print_main("Postprocessing physical quantities ...")
    x, y, y_pred = [], [], []
    for scenario in tqdm(test_data):
        x_i, y_i = scenario
        x.append(x_i)
        y.append(y_i)
        y_pred.append(model.net.predict(x_i))
    N = len(test_data)

## Space of initial conditions
Setting up

In [ ]:
if FLAGS['input_space']:
    path_ = path + '/initCond/'
    if not os.path.exists(path_):
        os.makedirs(path_)

file_train = case_dir + '/data/train/input_space.csv'
file_test = case_dir + '/data/test/input_space/data.csv'

inp_label = [
    r'$T$ [K]',
    r'$T_{{int}_0}$ [K]',
    r'$P_{_0}$ [Pa]',
    r'$X_{\text{O}_{2_0}}$'
]

In [ ]:
# Set up figure parameters ----------------------------------------------------
mpl_env = {
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'axes.labelsize': 15
}
# -----------------------------------------------------------------------------

style = dict(
    linestyle='',
    markersize=3
)

def plot_inp_space_3D(path, X_train, X_test):
    mpl.rcParams['axes.labelpad'] = 10
    fig = plt.figure()
    ax = plt.axes(projection='3d')
    ax.set_xlabel(inp_label[1])
    ax.set_ylabel(inp_label[2])
    ax.set_zlabel(inp_label[3])
    ax.plot3D(X_train[:,1], X_train[:,2], X_train[:,3], marker='x', c='k', **style)
    ax.plot3D(X_test[:,1], X_test[:,2], X_test[:,3], marker='o', c='r', fillstyle='full', **style)
    plt.grid()
    plt.show()
    fig.savefig(path+'/3D.pdf', dpi=600)

def plot_inp_space_2D(path, X_train, X_test, indeces=(0,1)):
    mpl.rcParams['axes.labelpad'] = 10
    i, j = indeces
    fig, ax = plt.subplots()
    ax.set_xlabel(inp_label[i])
    ax.set_ylabel(inp_label[j])
    ax.plot(X_train[:,i], X_train[:,j], marker='x', c='k', **style)
    ax.plot(X_test[:,i], X_test[:,j], marker='o', c='r', **style)
#     plt.show()
    fig.savefig(path+'/2D_'+str(j)+'.pdf')

if FLAGS['input_space']:
    utils.print_submain("Plotting input space")
    X_train = pd.read_csv(file_train, sep=',', header=0).to_numpy()
    X_test = pd.read_csv(file_test, sep=',', header=0, index_col=0).to_numpy()
    with plt.rc_context(mpl_env):
        plt.tight_layout()
        for indices in [(0,1), (2,3)]:
            plot_inp_space_2D(path_, X_train, X_test, indeces=indices)
        plot_inp_space_3D(path_, X_train, X_test)


## One-pass inference
Setting up

In [ ]:
if FLAGS['one_pass']:
    path_ = path + '/onePassInference/'
    if not os.path.exists(path_):
        os.makedirs(path_)

ts_label = []
for i, dim in enumerate(inputs.general.dim['out']):
    d = 1 if i == 0 else dim
    ts_label.append(
        [r'O'] + [r'O$_2^{({%s},{%s})}$' % (i+1,j+1) for j in range(d)]
    )               

In [ ]:
def plot_test(
    x,
    y,
    y_pred=None,
    var_names=None,
    fig_name=None,
    labels=[None, None],
    scales=['log', 'linear'],
    title=None,
    xlim=None,
    ylim=None,
    save=True,
    path=None,
    fig_ax=None,
    max_num_var=9
):
    if fig_ax is None:
        fig, ax = plt.subplots()
    else:
        fig, ax = fig_ax

    # Set up figure -----------------------------------------------------------
    if title is not None:
        ax.set_title(title)
    x_label, y_label = labels
    x_scale, y_scale = scales
    # X axis
    if x_label is not None:
        ax.set_xlabel(x_label)
    if x_scale is not None:
        ax.set_xscale(x_scale)
    if xlim is not None:
        ax.set_xlim(xlim)
    elif x_scale != 'log':
        ax.set_xlim([min(x), max(x)])
    # Y axis
    if y_label is not None:
        ax.set_ylabel(y_label)
    if y_scale == 'log':
        ax.set_yscale(y_scale)
        if ylim is None:
            ax.set_ylim([np.amin(y)*5.e-1, np.amax(y)*5.e+0])
    else:
        delta = np.amax(y)*0.1
        if ylim is None:
            ax.set_ylim([np.amin(y)-delta, np.amax(y)+delta])
    if ylim is not None:
        ax.set_ylim(ylim)
    # Check 2nd dim: looping over columns
    if len(y.shape) < 2:
        y = np.expand_dims(y, 1)

    # Plot figure -------------------------------------------------------------
    # Define styles
    true_style = dict(
        lw=1.
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=y.shape[0]//20,
        ls='--',
        lw=1.
    )
    # Define parameters
    n = y.shape[1]
    # Loop over features
    for d in range(n):
        # >> Plot true values
        ax.plot(
            x,
            y[:,d],
            c=colors[d],
            **true_style
        )
        # >> Plot predicted values
        ax.plot(
            x,
            y_pred[:,d],
            **pred_style,
            c=colors[d]
        )
    # Add legend
    if var_names is not None:
        idx = np.arange(0,n)
        if n > 6:
            if n > max_num_var:
                idx = np.sort(
                    train_test_split(
                        np.arange(n), test_size=int(max_num_var/n*10)/10
                    )[1]
                )
                idx[0], idx[-1]= 0, n-1
                n = max_num_var
            fs = 'small'
        else:
            fs = 'medium'
        ax.legend(
            [plt.plot([], [], c=colors[i])[0] for i in idx],
            [var_names[i] for i in idx],
            fontsize=fs,
            ncol=int(np.ceil(n/3))
        )

    # Save figure -------------------------------------------------------------
    if save:
        # Save figure
        name = y_scale if fig_name == None else fig_name + '_' + y_scale
        fig_path = path + '/' + name + '.pdf'
        fig.savefig(fig_path)
        
    if fig_ax is not None:
        return fig, ax

In [ ]:
def output_ops(y, timescale):
    if timescale == 0:
        return tf.reverse(y[0], axis=[1])
    else:
        return tf.concat([y[0][:,1:2], y[timescale]], 1)

def single_scenario(path, i):
    path_case = path_+'/case_'+str(i+1).zfill(2)
    for y_scale in ('linear', 'log'):
        path_y = path_case+'/'+y_scale
        if not os.path.exists(path_y):
            os.makedirs(path_y)
        for j, name in enumerate(ts_label):
            x_ = x[i][-1]
            y_ = output_ops(y[i], j)
            y_pred_ = output_ops(y_pred[i], j)
#             testing.plot_test(
            plot_test(
                x_,
                y_,
                y_pred_,
                var_names=name if j != len(ts_label)-1 else None,
                fig_name='TS'+str(j+1),
                labels=[r'$t$ [s]', r'$Y$'],
                scales=['log', y_scale],
                xlim=[1.e-12, 1.e-3],
                path=path_y
            )

if FLAGS['one_pass']:
    utils.print_submain("Testing 'one-pass' inference")
#     out = jl.Parallel(n_jobs=threads)(
#         jl.delayed(single_scenario)(path_, i) for i in tqdm(range(n))
#     )
    for i in tqdm(range(N)):
        single_scenario(path_, i)

In [ ]:
# def output_ops(y, timescale):
#     if timescale == 0:
#         return tf.reverse(y[0], axis=[1])
#     else:
#         return tf.concat([y[0][:,1:2], y[timescale]], 1)

# def single_scenario(path, x, y, y_pred, case):
#     path_case = path+'/'+case
#     for y_scale in ('linear', 'log'):
#         path_y = path_case+'/'+y_scale
#         if not os.path.exists(path_y):
#             os.makedirs(path_y)
#         for j, name in enumerate(inputs.network.names):
#             x_ = x[-1]
#             y_ = output_ops(y, j)
#             y_pred_ = output_ops(y_pred, j)
# #             testing.plot_test(
#             plot_test(
#                 path_y,
#                 x_,
#                 y_,
#                 y_pred_,
#                 ts_label[name],
#                 fig_name='TS'+str(i+1),
#                 labels=[r'$t$ [s]', r'$Y$'],
#                 scales=['log', y_scale]
#             )

# if FLAGS['one_pass']:
#     utils.print_submain("Testing 'one-pass' inference")
# #     out = jl.Parallel(n_jobs=threads)(
# #         jl.delayed(single_scenario)(path_, i) for i in tqdm(range(n))
# #     )
#     for i in tqdm(range(n)):
#         single_scenario(
#             path_, x[i], y[i], y_pred[i], 'case_'+str(index+1).zfill(2)
#         )

## Accuracy
Setting up

In [ ]:
if FLAGS['accuracy']:
    path_ = path + '/accuracy/'
    if not os.path.exists(path_):
        os.makedirs(path_)

# Num. of evaluation data
eval_pts = 1000

# Time limits
tlim = [1.e-12, 1.e-3]

# Norm order
order = 2

In [ ]:
def mre(order, axis=0, elementwise=False):
    def mean_rel_error(y_true, y_pred):
        if elementwise:
            err = norm((y_pred - y_true) / y_true, order)
            err = np.mean(err, axis=axis)
        else:
            err = np.linalg.norm(y_pred - y_true, ord=order, axis=axis)
            err = err / np.linalg.norm(y_true, ord=order, axis=axis)
        return err
    return mean_rel_error

def norm(x, order):
    x = np.absolute(x)
    if order == 1:
        return x
    else:
        return np.power(np.power(x, order), 1.0 / order)

In [ ]:
if FLAGS['accuracy']:
    utils.print_submain("Evaluating accuracy of the model")
    
    y_pred_, y_ = [], []
    for i in tqdm(range(len(y_pred))):
        mm = [np.argmin(np.absolute(x[i][1]-l)) for l in tlim]
        idx = np.sort(
            np.random.choice(
                np.arange(mm[0], mm[1]+1), eval_pts, replace=False
            )
        )
        y_.append(y[i][-1][idx])
        y_pred_.append(y_pred[i][-1][idx])
    y_pred_ = np.stack(y_pred_, 2)
    y_ = np.stack(y_, 2)

    err = 100*np.mean(
        mre(order, axis=0)(y_, y_pred_), axis=-1, keepdims=True
    )
    
    err_df = pd.DataFrame(
        err.T,
        columns=[f'Y_{i+1}' for i in range(len(err))],
        index=['err']
    )
    err_df = err_df.sort_values(by='err', axis=1, ascending=False)
    err_df.to_csv(path_+'norm.csv')


## Computational time
Setting up

In [ ]:
if FLAGS['comp_time']:
    path_ = path + '/comp_time/'
    if not os.path.exists(path_):
        os.makedirs(path_)

# Num. of input data
num_data = [int(10**i) for i in range(5)]

# Bars parameters
width = 0.1                # Bars width
loc = [-3., -1., 1., 3.]   # Bars locations
tick_loc = np.arange(len(num_data))   # x-ticks locations

# Num. of input data
evaluate = False
iteration = 100

In [ ]:
if FLAGS['comp_time']:
    utils.print_submain("Evaluating computational time")

    if evaluate:
        comp_time = []
        for ts in tqdm(range(4)):
            comp_time_ts = []
            for n in num_data:
                comp_time_ts_n = []
                for it in range(iteration):
                    x_ = [i[np.random.choice(i.shape[0], n, replace=True)] for i in x[-1]]
                    start = time.time()
        #             y_pred = model.net.predict(x_, max_ts=j)
                    y_pred = model.net.call(x_, max_ts=ts)
                    end = time.time()
                    nn_time = end - start
                    comp_time_ts_n.append(nn_time)
                comp_time_ts.append(np.mean(np.array(comp_time_ts_n)))
            comp_time.append(np.array(comp_time_ts))
        comp_time = np.vstack(comp_time)
        df = pd.DataFrame(
            comp_time,
            columns=num_data,
            index=['TS'+str(i+1) for i in range(4)]
        )
        df.to_csv(path_+'data.csv')
    else:
        comp_time = pd.read_csv(
            path_+'data.csv',
            header=0,
            index_col=0
        ).to_numpy()
    
    # Plot execution times
    fig, ax = plt.subplots()
    ax.set_xlabel('\# of input data')
    ax.set_xticks(tick_loc)
    ax.set_xticklabels(num_data)
    ax.set_ylabel('Time [s]')
    ax.set_yscale('log')
    for i, ct in enumerate(comp_time):
        ax.bar(tick_loc + width/2*loc[i], ct, width, label='TS'+str(i+1))
    ax.legend(fontsize=14, loc='best')
#     plt.show()
    fig.tight_layout()
    fig.savefig(path_ + 'times_bar.pdf')
    plt.close()

    # Plot style
    style = dict(
        marker='s',
        fillstyle='full',
        markersize=5,
        ls='--',
        lw=1.5
    )
    
    fig, ax = plt.subplots()
    ax.set_ylabel('Time [s]')
    ax.set_yscale('log')
    ax.set_xlabel('\# of input data')
    ax.set_xticks(tick_loc)
    ax.set_xticklabels(num_data)
    for i, ct in enumerate(comp_time):
        plt.plot(tick_loc, ct, label='TS'+str(i+1), **style)
    ax.legend(fontsize=14, loc='best')
#     plt.show()
    fig.tight_layout()
    fig.savefig(path_ + 'times_lines.pdf')
    plt.close()
    
    speedup = comp_time[-1] / comp_time[:3]
    
    fig, ax = plt.subplots()
    ax.set_ylabel('Speedup')
    ax.set_xlabel('\# of input data')
    ax.set_xticks(tick_loc)
    ax.set_xticklabels(num_data)
    for i, sp in enumerate(speedup):
        plt.plot(tick_loc, sp, label='TS'+str(i+1), **style)
    ax.legend(fontsize=14, loc='best')
#     plt.show()
    fig.tight_layout()
    fig.savefig(path_ + 'speedup.pdf')
    plt.close()


## Speedup
Setting up

In [ ]:
if FLAGS['speedup']:
    path_ = path + '/speedup/'
    if not os.path.exists(path_):
        os.makedirs(path_)

t_end = 1.e-7
num_ics = [int(10**i) for i in range(5)]
labels = ['DeepONet', 'Numerical Solver']

In [ ]:
if FLAGS['speedup']:
    utils.print_submain("Plotting execution time")

    # ODE solver
    dataGen.ODE_sol.t_end = t_end
    test_data = dataGen.testing_data(
        inputs.data.file['test'],
        exec_time=True,
        header=inputs.data.header
    )
    exec_time = []
    for scenario in test_data:
        exec_time.append(scenario[-1])
    ode_time_ref = np.mean(np.array(exec_time))

    # DeepONet & comparison
    exec_time = []
    for n in tqdm(num_ics):
        x_ = [i[np.random.choice(i.shape[0], n, replace=True)] for i in x[-1]]
        start = time.time()
        y_pred = model.net.predict(x_)
#         y_pred = model.net.call(x_)
        end = time.time()
        nn_time = end - start
        ode_time = ode_time_ref * n
        exec_time.append(np.array([nn_time, ode_time]))
    exec_time = np.vstack(exec_time)
    speedup = exec_time[:,1] / exec_time[:,0]

    # Plot style
    style = dict(
        marker='s',
        fillstyle='full',
        markersize=5,
        ls='--',
        lw=1.5
    )

    # Plot execution times
    fig = plt.figure()
    plt.xlabel('\# ODEs solved')
    plt.ylabel('Time [s]')
    plt.loglog(num_ics, exec_time, label=labels,**style)
    plt.legend(fontsize=16)
#     plt.show()
    fig.savefig(path_ + 'times.pdf')
    plt.close()

    # Plot speedup
    fig = plt.figure()
    plt.xlabel('\# ODEs solved')
    plt.ylabel('Speedup')
    plt.semilogx(num_ics, speedup, **style)
    fig.savefig(path_ + 'speedup.pdf')
#     plt.show()
    plt.close()

## Recursive inference
Setting up

In [ ]:
if FLAGS['recursive']:
    path_ = path + '/recursiveInference/'
    if not os.path.exists(path_):
        os.makedirs(path_)

Recursive inference solver

In [ ]:
def recursive_inference(
    ic,
    dt0=1.e-7,
    t_end=1.e-3,
    dt_str=1.4,
    dt_max=1.e-4
):
    # Initialize variables
    T = np.array([[0.]])
    Y = tf.concat(
        [1. - np.sum(ic[0:1,2:], keepdims=True), ic[0:1,2:]], axis=1
    )
    # Initialize loop
    t = T
    dt = dt0
    while t <= t_end:
        # Update inputs
        ic_ = np.expand_dims(tf.concat([ic[0,:2], Y[-1,1:]], axis=0), axis=0)
        t_ = np.array([[dt]])
        # Predictions
        y = output_ops(model.net([ic_,t_]), timescale=3)
        # Update time step
        t = t + dt
        dt = min(dt*dt_str, dt_max)
        # Collect data
        T = np.vstack((T, t))
        Y = np.vstack((Y, y))
    return T, Y

Recursive inference plotter

In [ ]:
def plot_recursive_inference(
    path,
    x,
    y,
    x_pred,
    y_pred,
    var_names=None,
    fig_name=None,
    labels=[r'$t$ [s]', r'$Y$'],
    scales=['log', 'linear'],
    xlim=None,
    ylim=None
):
    fig = plt.figure()

    # Set up figure
    x_label, y_label = labels
    x_scale, y_scale = scales

    # X axis
    if x_label is not None:
        plt.xlabel(x_label)
    if x_scale is not None:
        plt.xscale(x_scale)
    if xlim is not None:
        plt.xlim(xlim)
    else:
        plt.xlim([min(x_pred), max(x_pred)])

    # Y axis
    if y_label is not None:
        plt.ylabel(y_label)
    if y_scale == 'log':
        plt.yscale(y_scale)
        if ylim is None:
            plt.ylim([np.amin(y)*5.e-1, np.amax(y)*5.e+0])
    else:
        delta = np.amax(y)*0.1
        if ylim is None:
            plt.ylim([np.amin(y)-delta, np.amax(y)+delta])
    if ylim is not None:
        plt.ylim(ylim)

    if len(y.shape) < 2:
        y = np.expand_dims(y, 1)

    # Plot figure
    # Define styles
    true_style = dict(
        lw=1.
    )
    pred_style = dict(
        marker='x',
        fillstyle='none',
        markersize=5,
        ls='',
        lw=1.
    )

    # Define parameters
    colors = []
    n = y.shape[1]

    # Loop over features
    for d in range(n):

        # >> Plot true values
        plt.plot(
            x,
            y[:,d],
            **true_style
        )
        colors.append(plt.gca().lines[-1].get_color())

        # >> Plot predicted values
        plt.plot(
            x_pred,
            y_pred[:,d],
            **pred_style,
            label=var_names[d],
            c=colors[-1]
        )

    # Plot color legends
    if n > 12:
        fs = 'x-small'
    elif n > 6:
        fs = 'small'
    else:
        fs = 'medium'
    plt.legend(
        [plt.plot([], [], c=colors[i])[0] for i in range(len(var_names))],
        var_names,
        fontsize=fs,
        ncol=int(np.ceil(n/3))
    )

    # Save figure
    fig_path = path + '/' + fig_name + '.pdf'
    fig.savefig(fig_path)
    plt.close()

In [ ]:
if FLAGS['recursive']:
    utils.print_submain("Testing 'recursive' inference")
    for i in tqdm(range(N)):
        # Get solution
        t_ = x[i][1]
        y_ = output_ops(y[i], timescale=-1)
        t_pred_, y_pred_ = recursive_inference(x[i][0])
        # Plot solution
        path_case = path_+'/case_'+str(i+1).zfill(2)
        if not os.path.exists(path_case):
            os.makedirs(path_case)
        for y_scale in ('linear', 'log'):
            plot_recursive_inference(
                path_case,
                t_,
                y_,
                t_pred_,
                y_pred_,
                ts_label[-1],
                fig_name=y_scale,
                scales=['log', y_scale],
                xlim=[1.e-8, 1.e-3]
            )

## Nonequilibrium distance
A linear form for the distribution function, which results in a thermalized local Boltzmann distribution within individual bins, has been employed.

This function is defined as follows
\begin{equation*}
    \mathcal{F}_i\left(\epsilon_s\right):\qquad\log\left(\frac{g_s}{n_s}\right)=\alpha_i+\beta_i\epsilon_s
    \quad\forall s\in\mathcal{I}_i
\end{equation*}
where
\begin{align}
    \alpha_i&=\log\left(\frac{Q_i}{n_i}\right)\label{eq:alpha}\\
    \beta_i&=\frac{1}{k_BT_i}\nonumber\\
    n_i&=Y_i\frac{\rho}{m}\nonumber
\end{align}

Setting up

In [ ]:
if FLAGS['noneq_dist']:
    path_ = path + '/noneqDist/'
    if not os.path.exists(path_):
        os.makedirs(path_)

# Timescale to grouping strategies
grouping_list = ['RVE1', 'RVE3', 'RVE9', 'RVE27']

# Nonequilibrium distance names
dim = [1, 3, 9]
noneq_dist_names = [
    [r'$\epsilon^{({%s},{%s})}$' % (i+1, j+1) for j in range(dim[i])] for i in range(len(dim))
]

# Nonequilibrium distance calibration
mep_order = 0
norm_order = 2
mean = True

Initialize species thermodynamics

In [ ]:
def get_species_thermos(
    grouping_list,
    species
):
    species_thermos = []
    for grouping in grouping_list:
        # Update O2 with the proper grouping strategy
        sp = copy.deepcopy(species)
        sp['O2']['rovib']['grouping'] = grouping
        # Initialize species thermodynamics
        spTh = {
            name: specieMdl.specieThermos.SpecieThermos(
                name,
                info
            ) for name, info in sp.items()
        }
        # Add species thermodynamics to the corresponding timescale
        species_thermos.append(spTh)
    return species_thermos

speciesThermos = get_species_thermos(
    grouping_list,
    inputs.general.species
)

Initialize nonequilibrium controller

In [ ]:
from postprocess.src.noneqController import NoneqController

In [ ]:
noneqController = NoneqController(
    species_thermos=speciesThermos,
    ts_fast_conn=model.net.ts_fast_conn,
    mep_order=mep_order,
    norm_order=norm_order,
    mean=mean
)

### Nonequilibrium distance definition: OPTION 1
Consider the zero order truncation of the Maximum Entropy Principle reconstruction
\begin{align*}
    f_i=\alpha_i&=\log\left(\frac{Q_i}{n_i}\right)\\
    &=\log\left(\frac{Q_i}{Y_i}\frac{m}{\rho}\right)
\end{align*}

Nonequilibrium distance calculated as norm $\left(p=1\right)$ of the differential offset
* TS$^{(3)}$-to-TS$^{(4)}$
\begin{equation*}
    \epsilon_i^{(3-4)} = \sum_{j,\mathcal{I}_j^{(4)}\in\mathcal{I}_i^{(3)}}
    \left\lVert f_i^{(3)}-f_j^{(4)}\right\rVert_p
    \quad\forall\mathcal{I}_i^{(3)}\in\text{RVE}_9
\end{equation*}
* TS$^{(2)}$-to-TS$^{(4)}$
\begin{equation*}
    \epsilon_i^{(2-4)} = \sum_{j,\mathcal{I}_j^{(4)}\in\mathcal{I}_i^{(2)}}
    \left\lVert f_i^{(2)}-f_j^{(4)}\right\rVert_p
    \quad\forall\mathcal{I}_i^{(2)}\in\text{RVE}_3
\end{equation*}
* TS$^{(1)}$-to-TS$^{(4)}$
\begin{equation*}
    \epsilon_1^{(1-4)} = \sum_{j,\mathcal{I}_j^{(4)}\in\mathcal{I}_1^{(1)}}
    \left\lVert f_1^{(1)}-f_j^{(4)}\right\rVert_p
    \quad\mathcal{I}_1^{(1)}\in\text{RVE}_1
\end{equation*}

### Nonequilibrium distance definition: OPTION 2
Consider the first order truncation of the Maximum Entropy Principle reconstruction
\begin{align*}
    f_i(\epsilon_s)=\alpha_i+\beta_i\epsilon_s&=\log\left(\frac{Q_i}{n_i}\right)+\frac{\epsilon_s}{k_BT_i}\\
    &=\log\left(\frac{Q_i}{Y_i}\frac{m}{\rho}\right)+\frac{\epsilon_s}{k_BT_i}
\end{align*}

Nonequilibrium distance calculated as norm $\left(p=1\right)$ of the differential offset
* TS$^{(3)}$-to-TS$^{(4)}$
\begin{equation*}
    \epsilon_i^{(3-4)} = \sum_{j,\mathcal{I}_j^{(4)}\in\mathcal{I}_i^{(3)}}
    \left\lVert f_i(\epsilon_s)^{(3)}-f_j(\epsilon_s)^{(4)}\right\rVert_p
    \quad\wedge s\in\mathcal{I}_j^{(4)}
    \qquad\forall\mathcal{I}_i^{(3)}\in\text{RVE}_9
\end{equation*}
* TS$^{(2)}$-to-TS$^{(4)}$
\begin{equation*}
    \epsilon_i^{(2-4)} = \sum_{j,\mathcal{I}_j^{(4)}\in\mathcal{I}_i^{(2)}}
    \left\lVert f_i(\epsilon_s)^{(2)}-f_j(\epsilon_s)^{(4)}\right\rVert_p
    \quad\wedge s\in\mathcal{I}_j^{(4)}
    \qquad\forall\mathcal{I}_i^{(2)}\in\text{RVE}_3
\end{equation*}
* TS$^{(1)}$-to-TS$^{(4)}$
\begin{equation*}
    \epsilon_1^{(1-4)} = \sum_{j,\mathcal{I}_j^{(4)}\in\mathcal{I}_1^{(1)}}
    \left\lVert f_1(\epsilon_s)^{(1)}-f_j(\epsilon_s)^{(4)}\right\rVert_p
    \quad\wedge s\in\mathcal{I}_j^{(4)}
    \qquad\mathcal{I}_1^{(1)}\in\text{RVE}_1
\end{equation*}

Nonequilibrium distance plotter

In [ ]:
def plot_noneq_dist(
    x,
    y,
    x_pred=None,
    y_pred=None,
    var_names=None,
    fig_name=None,
    labels=[r'$t$ [s]', r'$\epsilon$'],
    scales=['log', 'linear'],
    title=None,
    xlim=None,
    ylim=None,
    save=True,
    path=None,
    fig_ax=None,
#     legend_loc='upper right'
):
    if fig_ax is None:
        fig, ax = plt.subplots()
    else:
        fig, ax = fig_ax

    # Set up figure -----------------------------------------------------------
    if title is not None:
        ax.set_title(title)
    x_label, y_label = labels
    x_scale, y_scale = scales
    # X axis
    if x_label is not None:
        ax.set_xlabel(x_label)
    if x_scale is not None:
        ax.set_xscale(x_scale)
    if xlim is not None:
        ax.set_xlim(xlim)
    elif x_scale != 'log':
        ax.set_xlim([min(x), max(x)])
    # Y axis
    if y_label is not None:
        ax.set_ylabel(y_label)
    if y_scale == 'log':
        ax.set_yscale(y_scale)
        if ylim is None:
            ax.set_ylim([np.amin(y)*5.e-1, np.amax(y)*5.e+0])
    else:
        delta = np.amax(y)*0.1
        if ylim is None:
            ax.set_ylim([np.amin(y)-delta, np.amax(y)+delta])
    if ylim is not None:
        ax.set_ylim(ylim)
    # Check 2nd dim: looping over columns
    if len(y.shape) < 2:
        y = np.expand_dims(y, 1)
#     ax.ticklabel_format(axis='y', style="sci", scilimits=(0,0))

    # Plot figure -------------------------------------------------------------
    # Define styles
    true_style = dict(
        lw=1.
    )
    pred_style = dict(
        marker='x',
        fillstyle='none',
        markersize=5,
        ls='',
        lw=1.
    )
    # Loop over columns
    n = y.shape[1]
    for d in range(n):
        # >> True values
        ax.plot(
            x,
            y[:,d],
            c=colors[d],
            **true_style
        )
        # >> Predicted values
        if y_pred is not None:
            ax.plot(
                x_pred,
                y_pred[:,d],
                label=var_names[d],
                c=colors[d],
                **pred_style,
            )
    # Add legend
    if var_names is not None:
        if n > 12:
            fs = 10
        elif n > 6:
            fs = 12
        else:
            fs = 14
        ax.legend(
            [plt.plot([], [], c=colors[i])[0] for i in range(len(var_names))],
            var_names,
            fontsize=fs,
            ncol=int(np.ceil(n/5)),
            loc='best'
#             loc=legend_loc
        )

    # Save figure -------------------------------------------------------------
    if save:
        # Save figure
        fig_path = path + '/' + fig_name + '.pdf'
        fig.savefig(fig_path)
        
    if fig_ax is not None:
        return fig, ax
            

In [ ]:
if FLAGS['noneq_dist']:
    utils.print_submain("Plotting nonequilibrium distance")
    for i in tqdm(range(N)):
        # Get solution
        t, noneq_dist = noneqController.eval_df_norm(
            data=copy.deepcopy(test_data[i])
        )
        # Plot nonequilibrium distance dynamics
        path_case = path_+'/case_'+str(i+1).zfill(2)
        if not os.path.exists(path_case):
            os.makedirs(path_case)
        for i, neqd in enumerate(noneq_dist):
            plot_noneq_dist(
                t,
                neqd,
                var_names=noneq_dist_names[i],
                path=path_case,
                fig_name='TS'+str(i+1),
            )


## Coarse-grained levels distribution
Setting up

In [ ]:
if FLAGS['levels']:
    path_ = path + '/levelsDist/'
    if not os.path.exists(path_):
        os.makedirs(path_)

# Get reactive mixture thermodynamics
ts_mixtures = [
    mixMdl.multiComponent.MultiComponent(spThs) \
        for spThs in speciesThermos
]

# Time instants to evaluate levels distribution
time_shots = [10.**(-i) for i in range(10,5,-2)]

Compute the levels distribution dynamics for each timescale

In [ ]:
def get_ts_lev_dist(data):
    # Extract data
    (ic, t), Y = data

    # Get temperature and density from initial condition
    T = np.take(ic, [0], axis=1)
    rho = np.take(ic, [1], axis=1)

    # Extract O and O2 mass fraction from Timescale 1
    Y_O = np.take(Y[0], [1], axis=1)
    Y[0] = np.take(Y[0], [0], axis=1)

    # Loop over the various timescale mixtures
    ts_lev_dist = []
    mix = dict(O=Y_O)
    for i, ts_mix in enumerate(ts_mixtures):
        mix['O2'] = Y[i]
        # Update the mixture with the new mass fractions
        ts_mix.update(mix)
        ts_mix.n_(rho=rho)
        # Update the partition functions
        for spTh in ts_mix.spTh.values():
            spTh.intPF.update(T)
        # Get n, g, E levels for each bin
        ts_lev_dist.append(
            ts_mix.spTh['O2'].intPF.levels_distribution()
        )
    return t, ts_lev_dist

Coarse-grained levels distribution plotter

In [ ]:
def plot_levels_dist(
    time=None,
    lev_dist=None,
    index=None,
    var_names=None,
    title=None,
    labels=[r'$\epsilon_i$ [eV]', r'$n_i\//\/g_i$ [m$^{-3}$]'],
    scales=['linear', 'log'],
    xlim=None,
    ylim=None,
    fig_ax=None,
    save=True,
    fig_path=None,
    col=None
):
    if fig_ax is None:
        fig, ax = plt.subplots()
    else:
        fig, ax = fig_ax

    # Set up figure -----------------------------------------------------------
    index = tf.nest.flatten(index)
    col = tf.nest.flatten(col)
    if title is not None:
        ax.set_title(title)
    x_label, y_label = labels
    x_scale, y_scale = scales
    # X axis
    if x_label is not None:
        ax.set_xlabel(x_label)
    if x_scale is not None:
        ax.set_xscale(x_scale)
    if xlim is not None:
        ax.set_xlim(xlim)
    # Y axis
    if y_label is not None:
        ax.set_ylabel(y_label)
    if x_scale is not None:
        ax.set_yscale(y_scale)
    if ylim is not None:
        ax.set_ylim(ylim)

    # Plot figure -------------------------------------------------------------
    style = dict(
        lw=1,
        marker='o',
        fillstyle='full',
        markersize=0.3,
        linestyle='none'
    )
    # Loop over Timescales
    for i, (n, g, E) in enumerate(lev_dist):
        # Loop over bins
        for b in range(len(n)):
            # Loop over time shots
            for j in range(len(index)):
                n_b = tf.reshape(n[b][index[j],:], (-1,1))
                c = i if len(index) == 1 else j
                ax.plot(E[b], n_b/g[b], c=col[c], **style)
    # Add legend
    if var_names is not None:
        ax.legend(
            [ax.plot([], [], c=col[i])[0] for i in range(len(var_names))],
            var_names,
            loc='lower left'
        )
    elif len(index) > 1:
        ax.legend(
            [ax.plot([], [], c=col[j])[0] for j in range(len(index))],
            [r'$t$ = %.1e s' % time[j] for j in index],
            loc='upper right'
        )
    # Time instant
    if time is not None and len(index) == 1:
        ax.text(
            0.75, 0.92,
            r'$t$ = %.1e s' % time[index],
            transform=ax.transAxes,
            fontsize=14
        )

    # Save figure -------------------------------------------------------------
    if save:
        fig.savefig(fig_path)
        
    if fig_ax is not None:
        return fig, ax


Plotting levels

In [ ]:
# Set up figure parameters ----------------------------------------------------
mpl_animation = {
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 16,
    'legend.fontsize': 14
}
# -----------------------------------------------------------------------------

def single_scenatio_lev(i, data):
    # Create folder
    path_test = path_+'/test_'+str(i+1).zfill(2)
    if not os.path.exists(path_test):
        os.makedirs(path_test)
    # Get levels distribution
    data = copy.deepcopy(data)
    t, ts_lev_dist = get_ts_lev_dist(data)
    # Multiple time shots in one figure
    idx = [np.argmin(np.absolute(t - t_i), axis=0)[0] for t_i in time_shots ]
    with plt.rc_context(mpl_animation):
        plot_levels_dist(
            time=t,
            lev_dist=[ts_lev_dist[-1]],
            index=idx,
            fig_path=path_test+'/TS4_t_shots.pdf',
            col=colors
        )
    # Loop over time instantsc
    for j, t_shot in enumerate(time_shots):
        path_tshots = path_test + '/shot_' + str(j+1)
        if not os.path.exists(path_tshots):
            os.makedirs(path_tshots)
        idx = np.argmin(np.absolute(t - t_shot), axis=0)[0]
        with plt.rc_context(mpl_animation):
            # Plot levels distribution
            plot_levels_dist(
#                 time=t,
                lev_dist=ts_lev_dist,
                index=idx,
                var_names=grouping_list,
                fig_path=path_tshots+'/TS.pdf',
                col=colors[:4]
            )
            for k in range(len(ts_lev_dist)):
                plot_levels_dist(
#                     time=t,
                    lev_dist=[ts_lev_dist[k]],
                    index=idx,
                    var_names=[grouping_list[k]],
                    fig_path=path_tshots+'/TS'+str(k+1)+'.pdf',
                    col=[colors[k]]
                )
                if k != 0:
                    plot_levels_dist(
#                         time=t,
                        lev_dist=ts_lev_dist[k-1:k+1],
                        index=idx,
                        var_names=grouping_list[k-1:k+1],
                        fig_path=path_tshots+'/TS'+str(k)+str(k+1)+'.pdf',
                        col=colors[k-1:k+1]
                    )
                if k != len(ts_lev_dist)-1:
                    plot_levels_dist(
#                         time=t,
                        lev_dist=[ts_lev_dist[k],ts_lev_dist[-1]],
                        index=idx,
                        var_names=[grouping_list[k],grouping_list[-1]],
                        fig_path=path_tshots+'/TS'+str(k+1)+'4.pdf',
                        col=[colors[k],colors[3]]
                    )

if FLAGS['levels']:
    utils.print_submain("Plotting levels distributions")
#     out = jl.Parallel(n_jobs=threads)(
#         jl.delayed(single_scenatio_lev)(i, data) for i, data in enumerate(tqdm(test_data))
#     )
    for i, data in enumerate(tqdm(test_data)):
        single_scenatio_lev(i, data)

## Animations of the dynamics
Setting up

In [ ]:
if FLAGS['animations']:
    path_ = path + '/animations/'
    if not os.path.exists(path_):
        os.makedirs(path_)
        
frames = 10

Sample one test case

In [ ]:
# k = np.random.randint(0, 100)
# data = copy.deepcopy(test_data[k])

Levels dynamics

In [ ]:
# t, ts_lev_dist = get_ts_lev_dist(data)

In [ ]:
# Set up figure parameters ----------------------------------------------------
mpl_animation = {
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 16,
    'legend.fontsize': 12
}
# -----------------------------------------------------------------------------

def init_lev_dist_lines(
    ax,
    var_names=grouping_list,
    title=None
):
    # Set up axes
    ax.set_xlabel(r'$\epsilon_i$ [eV]')
    ax.set_ylabel(r'$n_i\//\/g_i$ [m$^{-3}$]')
    # Define plot style
    style = dict(
        lw=1,
        marker='o',
        fillstyle='full',
        markersize=0.3,
        linestyle=''
    )
    # Add legend
    ax.legend(
        [ax.plot([], [], c=colors[i])[0] for i in range(len(var_names))],
        var_names,
        loc='lower left'
    )
    # Add title
    if title is not None:
        ax.set_title(title)
    # Initialize lines.Line2D objects
    lines = [
        ax.semilogy([], [], c=colors[i], **style)[0] \
            for i in range(len(var_names))
    ]
    return ax, lines

with plt.rc_context(mpl_animation):
    # Initialize a figure in which the graphs will be plotted
    fig, ax = plt.subplots()

    # Initialize levels distribution lines objects
    ax, lines = init_lev_dist_lines(ax)

    # Initialize text in ax
    t_text = ax.text(0.75, 0.92, '', transform=ax.transAxes, fontsize=14)

    def animate(frame):
        idx = int(frame/frames*len(t))
        # Write time instant
        t_text.set_text(r'$t$ = %.1e s' % t[idx])
        # Loop over Timescales
        for i, (n, g, E) in enumerate(ts_lev_dist):
            # Loop over bins
            x, y = [], []
            for b in range(len(n)):
                n_b = tf.reshape(n[b][idx,:], (-1,1))
                x.append(E[b])
                y.append(n_b/g[b])
            X, Y = list(map(lambda z: tf.concat(z, 0).numpy().squeeze(), [x, y]))
            lines[i].set_data(X.squeeze(), Y.squeeze())
        # Rescale axis limits
        ax.relim()
        ax.autoscale_view(tight=False)
        return lines

# # Get animation
# anim = FuncAnimation(
#     fig,
#     animate,
#     frames=frames,
#     blit=True
# )

# # Save animation
# anim.save(path_+'lev_dist.mp4', fps=10, dpi=600)

# # Display animation
# HTML(anim.to_jshtml())

## Adaptive technique
Setting up

In [ ]:
from src.chemistryModel import ChemistryModel
from src.adaptive import AdaptiveSolver

if FLAGS['adaptive']:
    path_ = path + '/adaptive/'
    if not os.path.exists(path_):
        os.makedirs(path_)

# Define noneq. metrics tolerances
tol = 0.1
tol_list = [0.01, 0.1, 0.5]

Analisys of grouped-reconstructed rate coefficients
\begin{align*}
    \frac{dn_P}{dt}&=
    -\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{PQ}n_Pn_\text{O}
    +\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{QP}n_Qn_\text{O}
    -K^D_{P}n_Pn_\text{O}
    +K^R_{P}n_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_P\subset\mathcal{I}\\
    %&&\text{Multiply by }\mathcal{F}_p\mathcal{F}_q:\\
    \mathcal{F}_p\mathcal{F}_q\frac{dn_P}{dt}&=
    \mathcal{F}_p\mathcal{F}_q\left(-\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{PQ}n_Pn_\text{O}
    +\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{QP}n_Qn_\text{O}
    -K^D_{P}n_Pn_\text{O}
    +K^R_{P}n_\text{O}^3\right)
    \qquad\forall\hspace{1mm}\mathcal{I}_P\subset\mathcal{I}\\
    %&&\text{Since }n_p&=n_P\mathcal{F}_p\text{ and }n_q=n_Q\mathcal{F}_q:\\
    \frac{dn_p}{dt}\mathcal{F}_q&=
    -\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{PQ}n_p\mathcal{F}_qn_\text{O}
    +\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{QP}\mathcal{F}_pn_qn_\text{O}
    -K^D_{P}\mathcal{F}_qn_pn_\text{O}
    +K^R_{P}\mathcal{F}_p\mathcal{F}_qn_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_p\subset\mathcal{I}\\
    %&&\text{Sum over }\mathcal{I}_q:\\
    \sum_{\mathcal{I}_q\subset\mathcal{I}_Q}\frac{dn_p}{dt}\mathcal{F}_q&=
    \sum_{\mathcal{I}_q\subset\mathcal{I}_Q}\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{PQ}n_p\mathcal{F}_qn_\text{O}
    +\sum_{\mathcal{I}_q\subset\mathcal{I}_Q}\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{QP}\mathcal{F}_pn_qn_\text{O}
    -\sum_{\mathcal{I}_q\subset\mathcal{I}_Q}K^D_{P}\mathcal{F}_qn_pn_\text{O}
    +\sum_{\mathcal{I}_q\subset\mathcal{I}_Q}K^R_{P}\mathcal{F}_p\mathcal{F}_qn_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_p\subset\mathcal{I}\\
    \frac{dn_p}{dt}&=
    -\sum_{\mathcal{I}_q\subset\mathcal{I}}K_{PQ}\mathcal{F}_qn_pn_\text{O}
    +\sum_{\mathcal{I}_q\subset\mathcal{I}}K_{QP}\mathcal{F}_pn_qn_\text{O}
    -K^D_{P}n_pn_\text{O}+K^R_{P}\mathcal{F}_pn_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_p\subset\mathcal{I}\\
    \frac{dn_p}{dt}&=
    -\sum_{\mathcal{I}_q\subset\mathcal{I}}\bar{K}_{pq}n_pn_\text{O}
    +\sum_{\mathcal{I}_q\subset\mathcal{I}}\bar{K}_{qp}n_qn_\text{O}
    -\bar{K}^D_{p}n_pn_\text{O}+\bar{K}^R_{p}n_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_p\subset\mathcal{I}\\
\end{align*}
So
\begin{align*}
    \bar{K}_{pq}&=K_{PQ}\mathcal{F}_q\\
    \bar{K}_{qp}&=K_{QP}\mathcal{F}_p\\
    \bar{K}^D_{p}&=K_P^D\\
    \bar{K}^R_{p}&=K_P^R\mathcal{F}_p
\end{align*}

Initialize chemistry models and adaptive solver

In [ ]:
chemModels = []
for i, spThs in enumerate(speciesThermos):
    pf = copy.deepcopy(inputs.general.processFlags)
    if i == 0:
        pf['excit'] = 0
    chemModels.append(
        ChemistryModel(
            specieThermos=spThs,
            processFlags=pf
        )
    )
    
adaptiveSolver = AdaptiveSolver(
    noneqController=noneqController,
    chemModels=chemModels
)

#### Custom ICs

In [ ]:
# Ttr = [14000., 8000.]
# T0 = [1000., 7500.]
# P0 = 8.e3
# XO2 = 0.95

# inp = {
#     'noneq': [Ttr[0], T0[0], P0, XO2],
#     'eq': [Ttr[1], T0[1], P0, XO2]
# }

Ttr = [12200., 8000.]
T0 = [2900., 8000.]
P0 = [4.6e3, 7.e3]
XO2 = [0.38, 0.6]

inp = {
    'noneq': [Ttr[0], T0[0], P0[0], XO2[0]],
    'eq': [Ttr[1], T0[1], P0[1], XO2[1]]
}

In [ ]:
if FLAGS['adaptive']:
    # Generating Initial Conditions
    rho_s = np.zeros((2,dataGen.num_bins+1), dtype=np.float64)
    for i, inp_i in enumerate(inp.values()):
        Ttr, T0, P0, XO2 = inp_i
        XO = 1. - XO2
        nd = P0 / (const.UKB * T0)
        Q_bins = dataGen.partition_func(T0)
        rho_s[i,:-1] = XO2 * Q_bins / sum(Q_bins) * 2.0 * const.mass['O'] * nd
        rho_s[i, -1] = XO  * const.mass['O'] * nd
    rho = np.sum(rho_s, axis=1, keepdims=True)
    Ttr = np.tile(utils.reshape_to_2DArray(Ttr), (2, 1))
    ic = np.hstack((Ttr, rho, np.divide(rho_s, rho)))

    # Evaluate exact solutions
    data_adpt = dataGen.testing_data(ic_test=ic, save=False)
    data_adpt = {k: data_adpt[i] for i, k in enumerate(inp.keys())}

    # Evaluate predicted solutions
    x_adpt, y_adpt, y_pred_adpt = [], [], []
    for k, dat in tqdm(data_adpt.items()):
        x_i, y_i = dat
        x_adpt.append(x_i)
        y_adpt.append(y_i)
        y_pred_adpt.append(model.net.predict(x_i))

In [ ]:
# %matplotlib inline
# for k, v in data_adpt.items():
#     plt.loglog(v[0][1][1:], v[1][0][:,1:2][1:], v[0][1][1:], v[1][-1][1:])
#     plt.show()

In [ ]:
# identifier = 'eq'

# data = tf.nest.map_structure(lambda x: x[1:], copy.deepcopy(custom_data[identifier]))
# data = copy.deepcopy(custom_data[identifier])

In [ ]:
# for i in range(len(x_adpt)):
#     x_ = x_adpt[i][-1]
#     y_ = output_ops(y_adpt[i], -1)
#     y_pred_ = output_ops(y_pred_adpt[i], -1)
#     plot_test(
#         x_,
#         y_,
#         y_pred_,
#         ts_label['TS3'],
#         fig_name='TS4',
#         labels=[r'$t$ [s]', r'$Y$'],
#         scales=['log', 'log'],
#         xlim=[1.e-12, 1.e-3],
#         save=False
#     )
#     plt.show()

Levels dynamics and nonequilibrium distance

In [ ]:
# # Evaluate noneq. distance
# t, noneq_dist = noneqController.eval_df_norm(
#     data=data
# )

In [ ]:
# # Set up figure parameters ----------------------------------------------------
# mpl_animation = {
#     'xtick.labelsize': 12,
#     'ytick.labelsize': 12,
#     'axes.labelsize': 12,
#     'axes.titlesize': 12,
#     'legend.fontsize': 8,
#     'figure.titlesize': 16,
#     'figure.figsize': np.array([6.4, 4.8])*1.5
# }
# # -----------------------------------------------------------------------------

# with plt.rc_context(mpl_animation):
#     # Initialize a figure with 4 subplots
#     fig, axs = plt.subplots(2,2)
#     axs = axs.reshape(-1)
#     plt.tight_layout()

#     # Plot nonequilibrium distance dynamics (3/4 subplots)
#     neqd_vlines = [] # Vertical lines for time evolution
#     for i, neqd in enumerate(noneq_dist):
#         axs[i] = plot_noneq_dist(
#             t,
#             neqd,
#             var_names=noneq_dist_names[i],
#             labels=[r'$t$ [s]', r'$\epsilon^{({%s}-4)}$' % (i+1)],
#             title=f'Nonequilibrium distance: TS{i+1}-TS4',
#             save=False,
#             fig_ax=(fig, axs[i])
#         )[1]
#         axs[i].ticklabel_format(axis='y', style="sci", scilimits=(0,0))
#         # Initialize vertical lines
#         neqd_vlines.append(
#             axs[i].axvline(0., *axs[i].get_ylim(), color='k', lw=0.5)
#         )

#     # Initialize levels distribution dynamics lines objects (4/4 subplots)
#     axs[-1], lev_lines = init_lev_dist_lines(
#         axs[-1],
#         title='States distribution dynamics'
#     )

#     def animate(frame):
#         idx = int(frame/frames*len(t))
#         # Write time instant
#         fig.suptitle(r'$t$ = %.1e s' % t[idx], fontsize=mpl_animation['figure.titlesize'])
#         # Set vertical lines data for time evolution
#         for vl in neqd_vlines:
#             vl.set_xdata(t[idx])
#         # Loop over Timescales
#         for i, (n, g, E) in enumerate(ts_lev_dist):
#             # Loop over bins
#             x, y = [], []
#             for b in range(len(n)):
#                 n_b = tf.reshape(n[b][idx,:], (-1,1))
#                 x.append(E[b])
#                 y.append(n_b/g[b])
#             X, Y = list(map(lambda z: tf.concat(z, 0).numpy().squeeze(), [x, y]))
#             lev_lines[i].set_data(X.squeeze(), Y.squeeze())
#         # Rescale axis limits
#         axs[-1].relim()
#         axs[-1].autoscale_view(tight=True)
#         return lev_lines + neqd_vlines

# # Get animation
# anim = FuncAnimation(
#     fig,
#     animate,
#     frames=frames,
#     blit=True
# )

# # Save animation
# anim.save(path_+'lev_neq_dist.mp4', fps=10, dpi=300)

# # Display animation
# HTML(anim.to_jshtml())

#### Part 1
Reconstructed groups & groups resolution for one tolerance value

In [ ]:
if FLAGS['adaptive']:

    adaptiveSolver.noneqController.tol = tol

    for i, name in enumerate(inp.keys()):

        # Define the output folder ------------------------------------------------
        path_case = path_+'/part1/case_'+name+'/'+str(float(noneqController.tol))+'/'
        if not os.path.exists(path_case):
            os.makedirs(path_case)

        # Apply adaptive technique ------------------------------------------------
        t, y_adpt_sol = adaptiveSolver.solve(
            x=copy.deepcopy(x_adpt[i]),
            Y=copy.deepcopy(y_adpt[i]),
    #         Y_pred=copy.deepcopy(y_adpt[i]),
            Y_pred=copy.deepcopy(y_pred_adpt[i])
        )

        # Plot Y evolution of reconstructed groups vs. exact Y --------------------
    #     for y_scale in ('linear', 'log'):
        plot_test(
            path=path_case,
            x=t,
            y=output_ops(y_adpt[i], -1),
            y_pred=y_adpt_sol,
            fig_name='TS4',
            labels=[r'$t$ [s]', r'$Y$'],
            scales=['log', 'log'],
            xlim=[1.e-12, 1.e-3]
        )

        # Plot groups resolution --------------------------------------------------
        adaptiveSolver.plot_group_res(
            path=path_case,
            xlim=[1.e-12, 1.e-3],
            fig_name='groups'
        )

        # Plot nonequilibrium distance dynamics -----------------------------------
        path_noneq = path_case+'/metrics/'
        if not os.path.exists(path_noneq):
            os.makedirs(path_noneq)
        for i, neqd in enumerate(adaptiveSolver.df_norm):
            plot_noneq_dist(
                t,
                neqd,
                var_names=noneq_dist_names[i],
                path=path_noneq,
                xlim=[1.e-12, 1.e-3],
                fig_name='TS'+str(i+1)
            )


#### Part 2
Mass fractions and number of groups needed

In [ ]:
def plot_res(
    t,
    y,
    exact=True,
    var_names=None,
    fig_name=None,
    labels=[r'$t$ [s]', r'$Y$'],
    scales=['log', 'linear'],
    title=None,
    xlim=None,
    ylim=None,
    save=True,
    path=None,
    fig_ax=None,
    max_num_var=9
):
    if fig_ax is None:
        fig, ax = plt.subplots()
    else:
        fig, ax = fig_ax

    # Set up figure -----------------------------------------------------------
    if title is not None:
        ax.set_title(title)
    x_label, y_label = labels
    x_scale, y_scale = scales
    # X axis
    if x_label is not None:
        ax.set_xlabel(x_label)
    if x_scale is not None:
        ax.set_xscale(x_scale)
    if xlim is not None:
        ax.set_xlim(xlim)
    elif x_scale != 'log':
        ax.set_xlim([min(x), max(x)])
    # Y axis
    if y_label is not None:
        ax.set_ylabel(y_label)
    if y_scale == 'log':
        ax.set_yscale(y_scale)
        if ylim is None:
            ax.set_ylim([np.amin(y)*5.e-1, np.amax(y)*5.e+0])
    else:
        delta = np.amax(y)*0.1
        if ylim is None:
            ax.set_ylim([np.amin(y)-delta, np.amax(y)+delta])
    if ylim is not None:
        ax.set_ylim(ylim)
    # Check 2nd dim: looping over columns
    if len(y.shape) < 2:
        y = np.expand_dims(y, 1)

    # Plot figure -------------------------------------------------------------
    # Define styles
    mrk = ['s','o','^']
    style = dict(
        fillstyle='full',
        markersize=3,
        markevery=y.shape[0]//20,
        lw=1.
    )
    # Define parameters
    n = y.shape[1]
    if exact:
        n = n - 1
    # Loop over features
    for d in range(n):
        ax.plot(
            t,
            y[:,d],
            marker=mrk[d],
            label=var_names[d],
            **style
        )
    if exact:
        ax.plot(
            t,
            y[:,-1],
            c='k',
            label=var_names[-1],
            linestyle='--',
            lw=1.
        )
    # Add legend
    ax.legend(
        fontsize=14,
        loc='best'
    )

    # Save figure -------------------------------------------------------------
    if save:
        # Save figure
        fig_path = path + '/' + fig_name + '.pdf'
        fig.savefig(fig_path)
        
    if fig_ax is not None:
        return fig, ax

In [ ]:
if FLAGS['adaptive']:
    for i, name in enumerate(inp.keys()):
        # Define the output folder ------------------------------------------------
        path_case = path_+'/part2/case_'+name+'/'
        if not os.path.exists(path_case):
            os.makedirs(path_case)

        resolution, mass, energy = [], [], []
        for j in range(len(tol_list)):
            adaptiveSolver.noneqController.tol = tol_list[j]

            # Apply adaptive technique ------------------------------------------------
            t, y_adpt_sol = adaptiveSolver.solve(
                x=copy.deepcopy(x_adpt[i]),
                Y=copy.deepcopy(y_adpt[i]),
                Y_pred=copy.deepcopy(y_adpt[i])
#                 Y_pred=copy.deepcopy(y_pred_adpt[i])
            )

            # Resolution
            resolution.append(
                np.sum(adaptiveSolver.group_res, axis=1, keepdims=True)
            )
            # Mass
            mass.append(1.-np.take(y_adpt_sol, [0], axis=1))
            # Energy
            # Get temperature and density (both constants)
            ic = x_adpt[i][0]
            T, rho = ic[0][:2]
            # Update thermos
            th = speciesThermos[-1]['O2'].thermo
            th.update(utils.reshape_to_2DArray(T))
            # Evaluate energy
            n_i = y_adpt_sol[:,1:] * rho / th.specie.M * const.UNA
#             Y_O2 = np.sum(y_adpt_sol[:,1:], axis=1, keepdims=True)
            e_int = th.e_int/const.EV_to_J*th.specie.m*n_i/np.sum(n_i, axis=1, keepdims=True)
            energy.append(np.sum(e_int, axis=1, keepdims=True))

        # Plot mass fraction evolution --------------------------------------------
        # Get exact value
        mass.append(np.take(y_adpt[i][0], [0], axis=1))
        plot_res(
            t,
            np.hstack(mass),
            var_names=[f'$\epsilon={t}$' for t in tol_list] + ['Exact'],
            fig_name='mass',
            xlim=[1.e-12, 1.e-3],
            path=path_case
        )

        # Plot energy evolution ---------------------------------------------------
        # Get exact value
        n_i = y_adpt[i][-1] * rho / th.specie.M * const.UNA
#         Y_O2 = np.sum(y_adpt[i][-1], axis=1, keepdims=True)
        e_int = th.e_int/const.EV_to_J*th.specie.m*n_i/np.sum(n_i, axis=1, keepdims=True)
        energy.append(np.sum(e_int, axis=1, keepdims=True))
        plot_res(
            t,
            np.hstack(energy),
            var_names=[f'$\epsilon={t}$' for t in tol_list] + ['Exact'],
            fig_name='energy',
            labels=[r'$t$ [s]', r'$E$ [eV]'],
            xlim=[1.e-12, 1.e-3],
            path=path_case
        )

        # Plot Y evolution of reconstructed groups vs. exact Y --------------------
        plot_res(
            t,
            np.hstack(resolution),
            exact=False,
            var_names=[f'$\epsilon={t}$' for t in tol_list],
            fig_name='resolution',
            labels=[r'$t$ [s]', r'\# of groups'],
            xlim=[1.e-12, 1.e-3],
            path=path_case,
        )


In [ ]:
# identifier = 'noneq'

# # data = tf.nest.map_structure(lambda x: x[1:], copy.deepcopy(custom_data[identifier]))
# data = copy.deepcopy(custom_data[identifier])

In [ ]:
# adaptiveSolver.atol = 1.e-15
# noneqController.tol = 1.e-3

In [ ]:
# data = copy.deepcopy(custom_data[identifier])
# data_adapt = adaptiveSolver.solve(data)
# print(tf.nest.map_structure(lambda x: x.shape, data_adapt))

In [ ]:
# data = copy.deepcopy(custom_data[identifier])
# x = data_adapt[0][1:]
# y = output_ops(data[1], -1)[1:]
# y_pred = data_adapt[1][1:]
# y_pred = np.hstack([np.take(y_pred, [-1], axis=1), np.delete(y_pred, -1, 1)])

# path_case = path_+'/case_'+identifier+'/'+str(float(noneqController.tol))+'/'
# if not os.path.exists(path_case):
#     os.makedirs(path_case)
                                              
# for y_scale in ('linear', 'log'):
#     plot_test(
#         path=path_case,
#         x=x,
#         y=y,
#         y_pred=y_pred,
#         var_names=ts_label['TS3'],
#         fig_name='TS4',
#         labels=[r'$t$ [s]', r'$Y$'],
#         scales=['log', y_scale],
#         xlim=[1.e-12, 1.e-3]
#     )

In [ ]:
# adaptiveSolver.plot_group_res(
#     path=path_case,
#     xlim=[1.e-12, 1.e-3],
#     var_names=grouping_list,
#     fig_name='groups'
# )

In [ ]:
# # Get solution
# t, noneq_dist = noneqController.eval_df_norm(
#     data=copy.deepcopy(custom_data[identifier])
# )
# # Plot nonequilibrium distance dynamics
# path_noneq = path_case+'/metrics/'
# if not os.path.exists(path_noneq):
#     os.makedirs(path_noneq)
# for i, neqd in enumerate(noneq_dist):
#     plot_noneq_dist(
#         t,
#         neqd,
#         var_names=noneq_dist_names[i],
#         path=path_noneq,
#         xlim=[1.e-12, 1.e-3],
#         fig_name='TS'+str(i+1),
#         legend_loc='upper right'
#     )

## Park vs Timescale 1
Setting up

In [ ]:
if FLAGS['park_ts1']:
    path_ = path + '/park_ts1/'
    if not os.path.exists(path_):
        os.makedirs(path_)

# Initialize conditions
T = 10000.
T0 = 2000.
P0 = 3000.
XO2 = 0.8

# Paths to solutions
parkQSS_file = cwd + '/src/park_ts1/parkQSS/output_box/box.dat'
park2001_file = cwd + '/src/park_ts1/park2001/output_box/box.dat'
sts_file = cwd + '/src/park_ts1/sts/excit_diss/output_box/box.dat'
pop_file = cwd + '/src/park_ts1/sts/excit_diss/output_box/pop_O2.dat'

# Labels
lbl = {
    'ts1': 'RVE27',
    'parkQSS': 'Park QSS',
    'park2001': 'Park 2001',
    'sts': 'StS'
}

In [ ]:
E_lev = np.array(speciesThermos[0]['O2'].specie.rv_lev['E']).squeeze()*const.UNA  # [J/mol]
g_lev = np.array(speciesThermos[0]['O2'].specie.rv_lev['g']).squeeze()
n_lev = len(speciesThermos[0]['O2'].specie.rv_lev['g'])

def X_rel_(instants):
    with open(pop_file, "r") as f:
        lines = f.readlines()
        del lines[:1]
    states = []
    for i in tqdm(range(instants)):
        states.append(np.loadtxt(lines[i*(n_lev+1)+1:(n_lev+1)*(i+1)], dtype=float).T[1].reshape(1,-1))
    n_states = np.vstack(states)*g_lev
    return n_states/np.sum(n_states, axis=1, keepdims=True)

In [ ]:
def extract_data(df):
    t = df[[col for col in df.columns if 't' in col]].to_numpy()
    X_O = df[[col for col in df.columns if col == 'X_O']].to_numpy()
    X_O2 = df[[col for col in df.columns if col == 'X_O2']].to_numpy()
    return t, X_O, X_O2

def Y_O2(X_O, X_O2):
    return 2*X_O2/(X_O + 2*X_O2)

def E_cg(rho, T, Y):
    # Update thermos
    speciesThermos[-1]['O2'].update(utils.reshape_to_2DArray(T))
    th = speciesThermos[-1]['O2'].thermo
    # Evaluate energy
    Y_O2 = np.sum(Y, axis=1, keepdims=True)
    return np.sum(th.e_int*rho*Y/Y_O2, axis=1, keepdims=True)

def E_park(rho, Tr, Tv):
    R = speciesThermos[0]['O2'].specie.R
    theta_v = speciesThermos[0]['O2'].specie.theta_v
    # Rotational energy
    E_r = R*Tr
    # Vibrational energy
    r = theta_v/Tv
    E_v = R*Tv*r/(np.exp(r)-1)
    return (E_r+E_v)*rho
#     return (E_v)*rho

def E_sts(rho, X_rel):
    M = speciesThermos[0]['O2'].specie.M
    return np.sum(X_rel * E_lev, axis=1)/M*rho

if FLAGS['park_ts1']:
    # Generating Initial Conditions
    rho_s = np.zeros((1,dataGen.num_bins+1), dtype=np.float64)
    XO = 1. - XO2
    nd = P0 / (const.UKB * T0)
    Q_bins = dataGen.partition_func(T0)
    rho_s[0,:-1] = XO2 * Q_bins / sum(Q_bins) * 2.0 * const.mass['O'] * nd
    rho_s[0, -1] = XO  * const.mass['O'] * nd
    rho = np.sum(rho_s, axis=1, keepdims=True)
    Ttr = utils.reshape_to_2DArray(T)
    ic = np.hstack((Ttr, rho, np.divide(rho_s, rho)))

    t, Y, E = {}, {}, {}
    # Solution from TS1
    data_TS1 = dataGen.testing_data(ic_test=ic, save=False)[0]
    t['ts1'] = data_TS1[0][1]
    Y['ts1'] = data_TS1[1][0][:,0]
    E['ts1'] = E_cg(rho, T, data_TS1[1][-1]).squeeze()
    
    # Solution from Park
    data_park = pd.read_csv(parkQSS_file, sep="  ", header=1, engine='python')
    Tv = data_park[
        [col for col in data_park.columns if 'Tv' in col]
    ].to_numpy()
#     Tr = data_park[
#         [col for col in data_park.columns if 'Tr' in col]
#     ].to_numpy()
    t['parkQSS'], X_O, X_O2 = extract_data(data_park)
    Y['parkQSS'] = Y_O2(X_O, X_O2)
    E['parkQSS'] = E_park(rho, T, Tv).squeeze()
    
    # Solution from Park
    data_park = pd.read_csv(park2001_file, sep="  ", header=1, engine='python')
    Tv = data_park[
        [col for col in data_park.columns if 'Tv' in col]
    ].to_numpy()
#     Tr = data_park[
#         [col for col in data_park.columns if 'Tr' in col]
#     ].to_numpy()
    t['park2001'], X_O, X_O2 = extract_data(data_park)
    Y['park2001'] = Y_O2(X_O, X_O2)
    E['park2001'] = E_park(rho, T, Tv).squeeze()
    
    # Solution from StS
    data_sts = pd.read_csv(sts_file, sep="  ", header=1, engine='python')
    t['sts'], X_O, X_O2 = extract_data(data_sts)
    Y['sts'] = Y_O2(X_O, X_O2)
    E['sts'] = E_sts(rho, X_rel_(len(Y['sts']))).squeeze()

In [ ]:
xscale = 'linear'
xlim = [1.e-14, 5.e-6]

if FLAGS['park_ts1']:
    fig, ax = plt.subplots()

    # Set up figure -----------------------------------------------------------
    # X axis
    ax.set_xlabel(r'$t$ [s]')
    ax.set_xscale(xscale)
    ax.set_xlim(xlim)
    # Y axis
    ax.set_ylabel(r'$Y$')

    # Plot figure -------------------------------------------------------------
    style = dict(
        marker='o',
        fillstyle='full',
        markersize=3,
        lw=1.
    )
    # Loop over features
#     stl = ['-','--',':']
    ax.plot(
        t['sts'],
        Y['sts'],
        linestyle='--',
        label=lbl['sts'],
        lw=1.,
        c='k'
    )
    for i, k in enumerate(['ts1', 'parkQSS', 'park2001']):
        ax.plot(
            t[k],
            Y[k],
#             linestyle=stl[i],
            label=lbl[k],
            lw=1.,
#             c='k',
#             markevery=Y[k].shape[0]//20,
#             **style
        )
    # Add legend
    ax.legend(
        fontsize=14,
        loc='best'
    )

    # Save figure -------------------------------------------------------------
    fig.savefig(path_ + f'/mass_{xscale}.pdf')


In [ ]:
if FLAGS['park_ts1']:
    fig, ax = plt.subplots()

    # Set up figure -----------------------------------------------------------
    # X axis
    ax.set_xlabel(r'$t$ [s]')
    ax.set_xscale('log')
    ax.set_xlim([1.e-8, 1.e-5])
    # Y axis
    ax.set_ylabel(r'$E$ [kJ m$^{-3}$]')

    # Plot figure -------------------------------------------------------------
    # Loop over features
    stl = ['-','--',':']
    for i, k in enumerate(['sts', 'park', 'ts1']):
        ax.plot(
            t[k],
            E[k],
            linestyle=stl[i],
            label=lbl[k],
            lw=1.,
            c='k'
        )
    # Add legend
    ax.legend(
        fontsize=14,
        loc='lower right'
    )

    # Save figure -------------------------------------------------------------
    fig.savefig(path_ + '/energy.pdf')
